# NanoStat-CTV: supervised fine-tuning (arm S0) on a free Colab T4, then evaluate

**What this does.** LoRA-fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` on programmatic **gold typed plans** for the pilot **train**
split (`training/sft_data.py`: registry-preferred method, CPU-verified assumption verdicts, no teacher LLM). It then serves base + adapter
with vLLM on `127.0.0.1` and evaluates on the **same dev split** as the B1 baselines in `docs/LLM_B1_BASELINE.md`, with the same harness
and prompt. Splits are group-exclusive: no dev or test task appears in training.

**Resilience.** Everything goes to Google Drive (`MyDrive/nanostat-ctv/sft`). Training checkpoints every 25 steps and resumes after a
disconnect. Evaluation traces are written per task and resume too. If the runtime drops, reconnect and **Run all** again.

**Runtime:** T4 GPU. Expect about 5 min of setup, 30–60 min of training (1 epoch, about 840 examples, not yet timed) and about 20 min of
evaluation. Report every result, including failures.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

if not os.path.exists("/content/nanostat-ctv"):
    !git clone -q https://github.com/Anandharajan/nanostat-ctv /content/nanostat-ctv
%cd /content/nanostat-ctv
!git pull -q; git log -1 --format='%h %s'
!pip -q install -e '.[gpu]'

In [ ]:
import pathlib

from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/nanostat-ctv/sft"
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
print("results ->", OUT)

## 1 · Data: pilot factory, SFT examples from the train split only

In [ ]:
!nanostat factory --profile pilot --out data 2>/dev/null | tail -2
!python -c "import hashlib,pathlib; p=pathlib.Path('data/tasks/dev.jsonl'); print('dev tasks', len(p.read_text().splitlines()), 'sha256', hashlib.sha256(p.read_bytes()).hexdigest()[:16])"
!python -c "from pathlib import Path; from training.sft_data import build_sft; print('SFT examples (train split):', build_sft(Path('data'), Path('{OUT}/sft_train.jsonl'), split='train'))"

## 2 · Supervised fine-tuning (LoRA r=8, fp16 autocast, resumable)

In [ ]:
!python -m training.train_gpu sft --data data --sft {OUT}/sft_train.jsonl --out {OUT}/s0 --seed 42 --epochs 1 --save-steps 25

In [ ]:
import json

log = json.loads(pathlib.Path(OUT, "s0", "train_log.json").read_text())
losses = [h["loss"] for h in log["log_history"] if "loss" in h]
print({k: v for k, v in log.items() if k != "log_history"})
print("loss first/last:", losses[:3], "...", losses[-3:])

## 3 · Serve base + adapter with vLLM, evaluate on the same dev split (prompt v3)

In [ ]:
!pip -q install vllm

In [ ]:
import subprocess
import time
import urllib.request

BASE = "http://127.0.0.1:8000/v1"
log = open("vllm_s0.log", "w")
server = subprocess.Popen(
    [
        "python",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "--served-model-name",
        "qwen2.5-coder-1.5b-instruct",
        "--enable-lora",
        "--max-lora-rank",
        "8",
        "--lora-modules",
        f"nanostat-s0={OUT}/s0/adapter",
        "--dtype",
        "half",
        "--max-model-len",
        "4096",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--seed",
        "0",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)
for _ in range(180):
    try:
        ids = [m["id"] for m in json.load(urllib.request.urlopen(BASE + "/models", timeout=2))["data"]]
        print("serving:", ids)
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM exited; see vllm_s0.log") from None
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not start in 15 min; see vllm_s0.log")

In [ ]:
from nanostat.prompting import PROMPT_VERSION, prompt_hash

PV = PROMPT_VERSION.removeprefix("typed-plan-")
print(PROMPT_VERSION, prompt_hash())
pathlib.Path(OUT, "eval").mkdir(exist_ok=True)

In [ ]:
!nanostat eval --arms B1 --llm-model nanostat-s0 --llm-base-url http://127.0.0.1:8000/v1 \
   --data data --split dev --max-steps 1 --traces {OUT}/eval/s0_{PV}_dev_steps1.jsonl \
   --out {OUT}/eval/s0_{PV}_dev_steps1.json > /dev/null
!nanostat eval --arms B1 --llm-model nanostat-s0 --llm-base-url http://127.0.0.1:8000/v1 \
   --data data --split dev --max-steps 3 --traces {OUT}/eval/s0_{PV}_dev_steps3.jsonl \
   --out {OUT}/eval/s0_{PV}_dev_steps3.json > /dev/null

In [ ]:
for f in sorted(pathlib.Path(OUT, "eval").glob("*.json")):
    r = json.loads(f.read_text())
    print(f.name, {k: v for k, v in r.items() if not isinstance(v, (dict, list))})

## 4 · Bring the result back (the adapter stays on Drive)

In [ ]:
!cp vllm_s0.log {OUT}/eval/ 2>/dev/null; (pip freeze | grep -E "^(vllm|torch|transformers|trl|peft)=="; nvidia-smi --query-gpu=name --format=csv,noheader; git log -1 --format=%h) > {OUT}/eval/versions_s0.txt
!cp {OUT}/s0/train_log.json {OUT}/eval/train_log_s0.json
!cd {OUT}/eval && rm -f /content/sft_s0_results.zip && zip -q /content/sft_s0_results.zip *
!ls -l /content/sft_s0_results.zip && sha256sum /content/sft_s0_results.zip
from google.colab import files

files.download("/content/sft_s0_results.zip")